# SignalPath evaluation pipeline

Runs reference WER/CER only when a real reference JSONL is supplied. Missing data is reported as pending, never fabricated.

In [ ]:
import json
import os
from pathlib import Path

from app.pipeline.benchmark import reference_metrics

input_path = os.getenv('SIGNALPATH_EVAL_JSONL')
output_path = Path('docs/eval/results.json')
if not input_path:
    result = {'status': 'pending', 'reason': 'Set SIGNALPATH_EVAL_JSONL to reference JSONL'}
else:
    rows = [json.loads(line) for line in Path(input_path).read_text(encoding='utf-8').splitlines() if line.strip()]
    metrics = [reference_metrics(row['reference'], row['raw'], row.get('corrected', row['raw'])) for row in rows]
    result = {'status': 'complete', 'sample_count': len(metrics), 'samples': metrics}
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(result, indent=2), encoding='utf-8')
print(json.dumps(result, indent=2))
